# 银行电话营销 XGBoost 分布式 Pipeline

## 架构
1. **特征配置**: `cat_features.txt` / `num_features.txt` / `label.txt` (一行一个特征)
2. **公共函数**: `pipeline_utils.py` (Spark、Pipeline、评估等)
3. **数据拆分**: 训练集 70% → 提取元信息 | 验证集 15% → Early Stopping | 测试集 15% → 最终评估

数据集: Bank Marketing (UCI, 葡萄牙银行定期存款营销)
全程 Spark DataFrame 分布式处理。

In [ ]:
import os, sys, time
sys.path.insert(0, '/home/feynman/work')
from pipeline_utils import *
from pyspark.sql import functions as F

In [ ]:
# 读取特征配置
BASE = "/home/feynman/work"
cat_feats, num_feats, label_col = load_feature_config(BASE)
print(f"Cat ({len(cat_feats)}): {cat_feats}")
print(f"Num ({len(num_feats)}): {num_feats}")
print(f"Label: {label_col}")

In [ ]:
# SparkSession
spark = create_spark_session("Bank-Marketing-Pipeline", shuffle_partitions=1)
spark

In [ ]:
# 加载数据 & 采样
DATA = os.path.join(BASE, "data/bank-additional/bank-additional-full.csv")
df = load_csv(spark, DATA, sample_size=800)
df = encode_label(df, label_col)
df = df.repartition(1).cache()
total = df.count()
pos = df.filter(F.col(label_col) == 1).count()
print(f"Data: {total} rows, pos={pos}, neg={total - pos}")
df.show(3)

In [ ]:
# 拆分为训练/验证/测试集
train_df, val_df, test_df = split_train_val_test(df, (0.7, 0.15, 0.15))
tc, vc, tec = train_df.count(), val_df.count(), test_df.count()
tp = train_df.filter(F.col(label_col) == 1).count()
sw = (tc - tp) / tp if tp else 1.0
print(f"Split: train={tc}, val={vc}, test={tec}")
print(f"Scale pos weight: {sw:.2f}")

In [ ]:
# 构建 & 拟合预处理 Pipeline
pipe = build_preprocessing_pipeline(cat_feats, num_feats)
print(f"Pipeline: {len(pipe.getStages())} stages")

t0 = time.time()
pm = pipe.fit(train_df)
print(f"Fitted: {time.time() - t0:.1f}s")

# Transform
train_pp = transform_and_cache(pm, train_df, label_col)
val_pp = transform_and_cache(pm, val_df, label_col)
test_pp = transform_and_cache(pm, test_df, label_col)
fdim = len(train_pp.select("features").first()["features"])
print(f"Feature dim: {fdim}")

In [ ]:
# 保存 Pipeline + 元信息到 S3
save_pipeline(pm)
save_metadata({
    "dataset": "Bank Marketing (UCI)",
    "cat_features": cat_feats, "num_features": num_feats,
    "stages": len(pipe.getStages()), "feature_dim": fdim,
    "train_count": tc, "val_count": vc, "test_count": tec,
})

In [ ]:
# XGBoost 训练 (Early Stopping on Validation)
from xgboost.spark import SparkXGBClassifier

xgb = SparkXGBClassifier(
    features_col="features", label_col=label_col,
    num_workers=2, n_estimators=80, max_depth=5,
    learning_rate=0.1, eval_metric="auc",
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=sw, missing=0.0, verbosity=1,
    validation_indicator_col="isVal",
    early_stopping_rounds=10,
)

# 合并训练+验证，标记验证集
combined = (train_pp.withColumn("isVal", F.lit(False))
           .union(val_pp.withColumn("isVal", F.lit(True)))) \
           .repartition(1).cache()
combined.count()

t0 = time.time()
model = xgb.fit(combined)
print(f"Training done: {time.time() - t0:.1f}s")

In [ ]:
# 测试集最终评估
test_preds = model.transform(test_pp).cache()
test_preds.count()
test_metrics = evaluate(test_preds, label_col)
print_metrics(test_metrics, prefix="TEST")

In [ ]:
# 保存模型
model.write().overwrite().save("s3a://models/xgboost_bank_marketing")
print("Model saved to S3")

# 特征重要性
print_feature_importance(model)

In [ ]:
# 推理验证：从 S3 加载 Pipeline + Model
from xgboost.spark import SparkXGBClassifierModel
pm2 = load_pipeline()
m2 = SparkXGBClassifierModel.load("s3a://models/xgboost_bank_marketing")

sample = test_df.limit(5).cache()
sample_preds = m2.transform(pm2.transform(sample))

for r in sample_preds.select("job","age","probability","prediction",label_col).collect():
    prob = float(r["probability"][1])
    pl, al = ("YES" if r["prediction"]==1 else "NO"), ("YES" if r[label_col]==1 else "NO")
    ok = "✓" if pl == al else "✗"
    print(f"  job={r['job']:<15} age={r['age']} prob={prob:.4f} pred={pl} actual={al} {ok}")

In [ ]:
# Summary
print(f"{'='*50}")
print(f"SUMMARY")
print(f"{'='*50}")
print(f"  Data:   {total} rows")
print(f"  Split:  train={tc} / val={vc} / test={tec}")
print(f"  Feats:  {len(cat_feats)} cat + {len(num_feats)} num → {fdim} dims")
print(f"  Stages: {len(pipe.getStages())}")
print(f"  TEST AUC: {test_metrics['auc_roc']:.4f}  F1: {test_metrics['f1']:.4f}  Acc: {test_metrics['accuracy']:.4f}")
print(f"{'='*50}")

# Cleanup
for d in [df, train_df, val_df, test_df, train_pp, val_pp, test_pp, test_preds, combined]:
    try: d.unpersist()
    except: pass
spark.stop()